# 04 — Model Comparison
This notebook evaluates the performance progression across the three implemented architectures.
It compares the point-estimators (LSTM, Transformer) against the probabilistic Autoformer.

In [1]:
import os, sys, numpy as np, pandas as pd, torch, torch.nn as nn, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from torch.utils.data import TensorDataset, DataLoader
sys.path.insert(0, os.path.abspath('..'))
DATA_PATH='../data'; SEQ_LEN=60; FORECAST=5; NUM_STOCKS=50; TRAIN_R=0.8; TGT=1; BATCH=64
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Ready | device:', device)


Ready | device: cpu


## Overall MAE & RMSE Comparison

In [2]:
import pandas as pd
rp='../outputs/results.csv'
if not os.path.exists(rp):
    print('Run notebooks 01, 02, 03 first.'); raise SystemExit
df=pd.read_csv(rp)
ov=df[df['day']=='Overall'][['model','mae','rmse']].copy()
ov.columns=['Model','MAE','RMSE']; ov=ov.sort_values('MAE').reset_index(drop=True)
print(ov.to_string(index=False))

colors={'LSTM':'#3498DB','Transformer':'#9B59B6','Autoformer+UQ':'#E67E22'}
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,metric in zip(axes,['MAE','RMSE']):
    bars=ax.bar(ov['Model'],ov[metric],color=[colors.get(m,'#95A5A6') for m in ov['Model']],edgecolor='white')
    ax.set_title(f'Overall {metric} by Model (lower is better)'); ax.set_ylabel(metric)
    for bar,val in zip(bars,ov[metric]):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.0002,f'{val:.4f}',ha='center',va='bottom',fontsize=9)
    ax.grid(axis='y',alpha=0.3)
plt.suptitle('Model Comparison: LSTM vs Transformer vs Autoformer+UQ',fontsize=13,fontweight='bold')
plt.tight_layout(); plt.savefig('../outputs/model_comparison.png',dpi=150); plt.close()
print('Comparison chart saved -> ../outputs/model_comparison.png')


        Model      MAE     RMSE
         LSTM 0.039595 0.059308
  Transformer 0.042599 0.061731
Autoformer+UQ 0.056731 0.074055
Comparison chart saved -> ../outputs/model_comparison.png


## Per-Day Comparison

In [3]:
per=df[df['day']!='Overall'].copy()
fig,axes=plt.subplots(1,2,figsize=(14,5))
for ax,metric in zip(axes,['mae','rmse']):
    for m,c in colors.items():
        sub=per[per['model']==m]
        if len(sub): ax.plot(sub['day'],sub[metric],marker='o',label=m,color=c,lw=2)
    ax.set_title(f'Per-Day {metric.upper()} — All Models'); ax.set_xlabel('Forecast Day'); ax.set_ylabel(metric.upper())
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('Progression: LSTM < Transformer < Autoformer+UQ',fontsize=13,fontweight='bold')
plt.tight_layout(); plt.savefig('../outputs/model_comparison_per_day.png',dpi=150); plt.close()
print('Per-day chart saved')


Per-day chart saved


## Summary
| Model | Architecture | Limitation |
|---|---|---|
| LSTM | Recurrent | Rigid point-estimator, lacks attention mechanism |
| Transformer | Self-Attention | Struggles with noisy continuous time-series |
| **Autoformer+UQ** | **Decomposition + Auto-Correlation + Dual Head** | **Computationally intensive** |